# Importação de Bibliotecas

In [ ]:
!pip install fundamentus
!pip install openpyxl
import fundamentus
import pandas as pd
import time

# Opcional: Configuração para o Pandas mostrar todas as colunas ao imprimir
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Iniciando a extração de dados da B3 (Data-base aproximada: 27/03/2026)...")


Iniciando a extração de dados da B3 (Data-base aproximada: 27/03/2026)...


# Coleta de dados e filtros de liquidez e preço

In [ ]:
# Baixa os múltiplos básicos de todas as ações da B3
df_mercado = fundamentus.get_resultado()

# Aplicando os filtros solicitados:
# 1. Cotação menor ou igual a R$ 20,00
# 2. Liquidez diária (média de 2 meses) maior ou igual a R$ 50.000.000
filtro = (df_mercado['cotacao'] <= 20.00) & (df_mercado['liq2m'] >= 50000000)

# Cria um novo DataFrame apenas com as empresas que passaram no filtro
df_filtrado = df_mercado[filtro].copy()

print(f"Filtro aplicado. Restaram {len(df_filtrado)} empresas com alta liquidez e cotação até R$ 20,00.")

Filtro aplicado. Restaram 42 empresas com alta liquidez e cotação até R$ 20,00.


# ENRIQUECIMENTO DE DADOS (SEGMENTO E ESTRUTURA DE CAPITAL)

In [ ]:
# 1. Carrega o arquivo EXCEL diretamente (já que o CSV não foi encontrado)
# O parâmetro usecols="D,F" pega exatamente a Coluna D e Coluna F
# Instalamos openpyxl caso necessário para ler .xlsx
try:
    df_b3 = pd.read_excel('ClassifSetorial.xlsx', skiprows=2, usecols="D,F")
except:
    !pip install openpyxl
    df_b3 = pd.read_excel('ClassifSetorial.xlsx', skiprows=2, usecols="D,F")

df_b3.columns = ['Segmento', 'Codigo_B3']

# 2. Tratamento das células mescladas do Excel
# O método ffill() preenche os vazios "puxando" o nome do segmento da linha de cima
df_b3['Segmento'] = df_b3['Segmento'].ffill()

# 3. Limpeza dos códigos e padronização para texto em maiúsculo
df_b3 = df_b3.dropna(subset=['Codigo_B3'])
df_b3['Codigo_B3'] = df_b3['Codigo_B3'].astype(str).str.strip().str.upper()

# 4. Prepara a base do Fundamentus (df_filtrado) para o cruzamento
df_final = df_filtrado.reset_index().rename(columns={'papel': 'Ticker da Ação'})

# Extraímos os 4 primeiros caracteres do Ticker (ex: de 'ABEV3' para 'ABEV')
# para criar a chave de ligação idêntica à da planilha da B3
df_final['Codigo_B3'] = df_final['Ticker da Ação'].str[:4]

# 5. Realiza o cruzamento (Left Join)
df_final = pd.merge(df_final, df_b3, on='Codigo_B3', how='left')

# Remove a coluna auxiliar do Código de 4 letras que não será mais usada
df_final = df_final.drop(columns=['Codigo_B3'])

print(f"Enriquecimento concluído. {len(df_final)} ações processadas.")

Enriquecimento concluído. 42 ações processadas.


# SELEÇÃO DE COLUNAS, FORMATAÇÃO ESTÉTICA E RESULTADO FINAL

In [ ]:
# O dicionário abaixo dita a ordem exata das colunas.
# Repare que 'Segmento' é a segunda chave do dicionário.
colunas_selecionadas = {
    'Ticker da Ação': 'Ticker da Ação',
    'Segmento': 'Segmento',
    'cotacao': 'Cotação',
    'pl': 'P/L',
    'dy': 'DY',
    'roic': 'ROIC',
    'roe': 'ROE',
    'mrgliq': 'Margem Líquida',
    'divbpatr': 'Dívida Bruta/PL'
}

df_projeto = df_final[list(colunas_selecionadas.keys())].rename(columns=colunas_selecionadas)

# --- APLICAÇÃO DAS FORMATAÇÕES SOLICITADAS ---

# 1. Cotação em formato financeiro (R$)
df_projeto['Cotação'] = df_projeto['Cotação'].apply(lambda x: f"R$ {x:,.2f}".replace('.', ','))

# 2. Indicadores de Rentabilidade em Percentual (%)
colunas_percentuais = ['DY', 'ROIC', 'ROE', 'Margem Líquida']
for col in colunas_percentuais:
    # Caso ocorra algum erro na extração, evitamos que o código quebre
    df_projeto[col] = pd.to_numeric(df_projeto[col], errors='coerce').fillna(0)
    df_projeto[col] = df_projeto[col].apply(lambda x: f"{x:.2%}".replace('.', ','))

# 3. Indicadores de Estrutura de Capital e Preço em formato de Múltiplo (x)
colunas_multiplos = ['P/L', 'Dívida Bruta/PL']
for col in colunas_multiplos:
    df_projeto[col] = pd.to_numeric(df_projeto[col], errors='coerce').fillna(0)
    df_projeto[col] = df_projeto[col].apply(lambda x: f"{x:.1f}x".replace('.', ','))

print("\n" + "="*100)
print("SCREENING FINALIZADO: EMPRESAS SELECIONADAS (COM SEGMENTO B3)")
print("="*100)
print(df_projeto)


SCREENING FINALIZADO: EMPRESAS SELECIONADAS (COM SEGMENTO B3)
   Ticker da Ação                                  Segmento   Cotação     P/L      DY    ROIC       ROE Margem Líquida Dívida Bruta/PL
0           ABEV3                  Cervejas e Refrigerantes  R$ 14,76   15,0x   6,68%  19,81%    17,63%         18,12%            0,0x
1           ASAI3                                 Alimentos   R$ 8,69   23,7x   1,37%  19,52%     8,95%          0,64%            2,9x
2           B3SA3             Serviços Financeiros Diversos  R$ 17,21   18,9x   3,22%  20,37%    26,29%         41,24%            0,9x
3           BBDC3                                    Bancos  R$ 16,23    7,0x   8,04%   0,00%    14,25%          0,00%            0,0x
4           BBDC4                                    Bancos  R$ 18,52    8,0x   7,75%   0,00%    14,25%          0,00%            0,0x
5           BEEF3                        Carnes e Derivados   R$ 4,07    5,0x   4,04%  17,90%    61,86%          1,55%         

# EXPORTAÇÃO DO DATAFRAME EM PLANILHA

In [ ]:
# Definindo o nome do arquivo que será salvo
nome_arquivo = 'screening_b3_resultado.xlsx'

# Exportando para Excel
# O parâmetro index=False evita que os números das linhas (0, 1, 2...) virem uma coluna inútil no Excel
df_projeto.to_excel(nome_arquivo, index=False)

print(f"\nSucesso! Os dados foram salvos na planilha: {nome_arquivo}")

# Se preferir exportar em formato CSV (mais leve), você pode usar a linha abaixo em vez da de cima:
# df_projeto.to_csv('screening_b3_resultado.csv', sep=';', decimal=',', index=False, encoding='utf-8-sig')


Sucesso! Os dados foram salvos na planilha: screening_b3_resultado.xlsx
